# Bilprisprediktion med ML

**Uppgift:** Skapa en ML-modell för bilprisprediktion och en Streamlit-app.

I detta script demonstrerar vi hur en Machine Learning workflow kan se ut när man använder train, validation och test set.

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib

print(" Imports klara!")

 Imports klara!


## Ladda data

In [2]:
# Ladda dataset
df = pd.read_csv('car_price_dataset.csv', sep=';')
print(f"Dataset: {df.shape[0]:,} rader, {df.shape[1]} kolumner")
print(f"Kolumner: {list(df.columns)}")
print("\nFörsta 3 rader:")
print(df.head(3))

Dataset: 10,000 rader, 10 kolumner
Kolumner: ['Brand', 'Model', 'Year', 'Engine_Size', 'Fuel_Type', 'Transmission', 'Mileage', 'Doors', 'Owner_Count', 'Price']

Första 3 rader:
       Brand   Model  Year  Engine_Size Fuel_Type Transmission  Mileage  \
0        Kia     Rio  2020          4.2    Diesel       Manual   289944   
1  Chevrolet  Malibu  2012          2.0    Hybrid    Automatic     5356   
2   Mercedes     GLA  2020          4.2    Diesel    Automatic   231440   

   Doors  Owner_Count  Price  
0      3            5   8501  
1      2            3  12092  
2      4            2  11171  


## Förbereda data på det sätt som kommer användas

In [3]:
# Separera features och target
X = df.drop('Price', axis=1)  # Features (oberoende variabler)
y = df['Price']               # Target (beroende variabel)

print("X info:")
print(X.info())
print("\ny info:")
print(y.info())

X info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Brand         10000 non-null  object 
 1   Model         10000 non-null  object 
 2   Year          10000 non-null  int64  
 3   Engine_Size   10000 non-null  float64
 4   Fuel_Type     10000 non-null  object 
 5   Transmission  10000 non-null  object 
 6   Mileage       10000 non-null  int64  
 7   Doors         10000 non-null  int64  
 8   Owner_Count   10000 non-null  int64  
dtypes: float64(1), int64(4), object(4)
memory usage: 703.3+ KB
None

y info:
<class 'pandas.core.series.Series'>
RangeIndex: 10000 entries, 0 to 9999
Series name: Price
Non-Null Count  Dtype
--------------  -----
10000 non-null  int64
dtypes: int64(1)
memory usage: 78.3 KB
None


## Train, Validation och Test Set

In [4]:
# Första uppdelning: train+validation vs test (80% vs 20%)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Andra uppdelning: train vs validation (70% vs 30% av train_full)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.3, random_state=42)

print(f"Träningsdata: {X_train.shape[0]:,} rader")
print(f"Valideringsdata: {X_val.shape[0]:,} rader")
print(f"Testdata: {X_test.shape[0]:,} rader")
print(f"Features: {X_train.shape[1]} kolumner")

Träningsdata: 5,600 rader
Valideringsdata: 2,400 rader
Testdata: 2,000 rader
Features: 9 kolumner


## EDA

Vi får bara utforska/lära oss saker på träningsdata när vi skapar vår modell.

In [5]:
# Grundläggande statistik på träningsdata
print("Beskrivande statistik (träningsdata):")
print(X_train.describe())

print("\nKategoriska variabler (träningsdata):")
categorical_cols = X_train.select_dtypes(include=['object']).columns
for col in categorical_cols:
    print(f"{col}: {X_train[col].nunique()} unika värden")

print(f"\nPrisstatistik (träningsdata):")
print(f"Medelpris: ${y_train.mean():,.0f}")
print(f"Minpris: ${y_train.min():,.0f}")
print(f"Maxpris: ${y_train.max():,.0f}")

Beskrivande statistik (träningsdata):
              Year  Engine_Size        Mileage        Doors  Owner_Count
count  5600.000000  5600.000000    5600.000000  5600.000000  5600.000000
mean   2011.650714     3.001821  149284.445179     3.494464     2.972500
std       6.899047     1.156162   86363.925376     1.109622     1.421128
min    2000.000000     1.000000      25.000000     2.000000     1.000000
25%    2006.000000     2.000000   74547.000000     3.000000     2.000000
50%    2012.000000     3.000000  148703.500000     3.000000     3.000000
75%    2018.000000     4.000000  224348.250000     4.000000     4.000000
max    2023.000000     5.000000  299714.000000     5.000000     5.000000

Kategoriska variabler (träningsdata):
Brand: 10 unika värden
Model: 30 unika värden
Fuel_Type: 4 unika värden
Transmission: 3 unika värden

Prisstatistik (träningsdata):
Medelpris: $8,872
Minpris: $2,000
Maxpris: $18,301


## Pipeline-klass

In [6]:
# Enkel pipeline för bilprisprediktion
class CarPipeline:
    def __init__(self):
        self.model = RandomForestRegressor(n_estimators=100, random_state=42)
        self.brand_encoder = OneHotEncoder(drop='first', sparse_output=False)
        self.model_encoder = OneHotEncoder(drop='first', sparse_output=False)
        self.fuel_encoder = OneHotEncoder(drop='first', sparse_output=False)
        self.transmission_encoder = OneHotEncoder(drop='first', sparse_output=False)
        self.scaler = StandardScaler()
        
    def fit(self, X, y):
        X_processed = X.copy()
        
        # Feature engineering: skapa nya features
        X_processed['Age'] = 2024 - X_processed['Year']
        X_processed['Miles_Per_Year'] = X_processed['Mileage'] / (X_processed['Age'] + 1)
        
        # Träna encoders på kategoriska variabler
        self.brand_encoder.fit(X_processed[['Brand']])
        self.model_encoder.fit(X_processed[['Model']])
        self.fuel_encoder.fit(X_processed[['Fuel_Type']])
        self.transmission_encoder.fit(X_processed[['Transmission']])
        
        # Transformera kategoriska variabler till dummy variables
        brand_encoded = self.brand_encoder.transform(X_processed[['Brand']])
        model_encoded = self.model_encoder.transform(X_processed[['Model']])
        fuel_encoded = self.fuel_encoder.transform(X_processed[['Fuel_Type']])
        transmission_encoded = self.transmission_encoder.transform(X_processed[['Transmission']])
        
        # Skala numeriska features
        numeric_features = ['Year', 'Mileage', 'Engine_Size', 'Doors', 'Owner_Count', 'Age', 'Miles_Per_Year']
        X_numeric = X_processed[numeric_features]
        X_scaled = self.scaler.fit_transform(X_numeric)
        
        # Kombinera alla features
        X_combined = np.hstack([X_scaled, brand_encoded, model_encoded, fuel_encoded, transmission_encoded])
        
        # Träna modellen
        self.model.fit(X_combined, y)
        return self
    
    def predict(self, X):
        X_processed = X.copy()
        
        # Samma feature engineering som vid träning
        X_processed['Age'] = 2024 - X_processed['Year']
        X_processed['Miles_Per_Year'] = X_processed['Mileage'] / (X_processed['Age'] + 1)
        
        # Transformera kategoriska variabler
        brand_encoded = self.brand_encoder.transform(X_processed[['Brand']])
        model_encoded = self.model_encoder.transform(X_processed[['Model']])
        fuel_encoded = self.fuel_encoder.transform(X_processed[['Fuel_Type']])
        transmission_encoded = self.transmission_encoder.transform(X_processed[['Transmission']])
        
        # Skala numeriska features
        numeric_features = ['Year', 'Mileage', 'Engine_Size', 'Doors', 'Owner_Count', 'Age', 'Miles_Per_Year']
        X_numeric = X_processed[numeric_features]
        X_scaled = self.scaler.transform(X_numeric)
        
        # Kombinera och prediktera
        X_combined = np.hstack([X_scaled, brand_encoded, model_encoded, fuel_encoded, transmission_encoded])
        return self.model.predict(X_combined)

print(" Pipeline-klass skapad!")

 Pipeline-klass skapad!


## Träna modell på träningsdata

In [7]:
# Träna pipeline på träningsdata
pipeline = CarPipeline()
pipeline.fit(X_train, y_train)

print(" Modell tränad på träningsdata!")

 Modell tränad på träningsdata!


## Välja bästa modell genom valideringsset

In [8]:
# Prediktera på valideringsdata
y_val_pred = pipeline.predict(X_val)

# Beräkna RMSE på valideringsdata
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))

print(f"RMSE på valideringsdata: ${val_rmse:,.0f}")
print(f"Genomsnittligt fel: ${val_rmse:,.0f}")

# Visa några exempel
print("\nExempel på valideringsdata:")
for i in range(5):
    print(f"Prediktion: ${y_val_pred[i]:,.0f}, Faktiskt: ${y_val.iloc[i]:,.0f}")

RMSE på valideringsdata: $633
Genomsnittligt fel: $633

Exempel på valideringsdata:
Prediktion: $8,505, Faktiskt: $8,226
Prediktion: $12,396, Faktiskt: $13,701
Prediktion: $5,252, Faktiskt: $4,486
Prediktion: $6,133, Faktiskt: $5,225
Prediktion: $9,594, Faktiskt: $8,891


## Jämförelse av modeller

Låt oss jämföra Random Forest med Linear Regression.

In [9]:
# Jämförelse av Random Forest vs Linear Regression
print("Jämförer Random Forest vs Linear Regression...")

# Random Forest (redan tränad)
rf_val_rmse = val_rmse

# Linear Regression
lr_pipeline = CarPipeline()
lr_pipeline.model = LinearRegression()  # Byt till Linear Regression
lr_pipeline.fit(X_train, y_train)

# Prediktera med Linear Regression
y_val_pred_lr = lr_pipeline.predict(X_val)
lr_val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred_lr))

# Visa resultat
print(f"Random Forest RMSE: ${rf_val_rmse:,.0f}")
print(f"Linear Regression RMSE: ${lr_val_rmse:,.0f}")

if rf_val_rmse < lr_val_rmse:
    print(" Random Forest presterar bättre!")
    best_model = "Random Forest"
    best_pipeline = pipeline
else:
    print(" Linear Regression presterar bättre!")
    best_model = "Linear Regression"
    best_pipeline = lr_pipeline

print(f"Väljer {best_model} som bästa modell.")

Jämförer Random Forest vs Linear Regression...
Random Forest RMSE: $633
Linear Regression RMSE: $72
 Linear Regression presterar bättre!
Väljer Linear Regression som bästa modell.


## Utvärdera vald modell genom testset

Nu tränar vi om vår bästa modell på train + validation data.

In [10]:
# Träna om den bästa modellen på hela träningsdata (train + validation)
pipeline_final = CarPipeline()
if best_model == "Linear Regression":
    pipeline_final.model = LinearRegression()
pipeline_final.fit(X_train_full, y_train_full)

# Prediktera på testdata
y_test_pred = pipeline_final.predict(X_test)

# Beräkna RMSE på testdata
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print(f"RMSE på testdata: ${test_rmse:,.0f}")
print(f"Genomsnittligt fel: ${test_rmse:,.0f}")

# Visa några exempel
print("\nExempel på testdata:")
for i in range(5):
    print(f"Prediktion: ${y_test_pred[i]:,.0f}, Faktiskt: ${y_test.iloc[i]:,.0f}")

RMSE på testdata: $65
Genomsnittligt fel: $65

Exempel på testdata:
Prediktion: $1,692, Faktiskt: $2,000
Prediktion: $11,150, Faktiskt: $11,164
Prediktion: $14,602, Faktiskt: $14,630
Prediktion: $7,349, Faktiskt: $7,334
Prediktion: $10,146, Faktiskt: $10,127


## Spara modell för Streamlit-app

In [11]:
# Spara den slutliga modellen
joblib.dump(pipeline_final, 'car_pipeline.pkl')
print(" Modell sparad som car_pipeline.pkl")

# Testa sparad modell
loaded_pipeline = joblib.load('car_pipeline.pkl')
test_sample = X_test.iloc[0:1]
prediction = loaded_pipeline.predict(test_sample)[0]
actual = y_test.iloc[0]
print(f"\nTest: Prediktion ${prediction:,.0f}, Faktiskt ${actual:,.0f}")
print(" Klar för Streamlit!")

 Modell sparad som car_pipeline.pkl

Test: Prediktion $1,692, Faktiskt $2,000
 Klar för Streamlit!
